<a href="https://colab.research.google.com/github/fvarellalopes/clawsouls/blob/main/Z_Image_Turbo_4bit_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U git+https://github.com/huggingface/diffusers git+https://github.com/Disty0/sdnq

  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-ol3d7wik
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-ol3d7wik
  Resolved https://github.com/huggingface/diffusers to commit 48f39c2d59e8db444cb37f91e72413a1db9a2dd6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/Disty0/sdnq to /tmp/pip-req-build-47u0yvzs
  Running command git clone --filter=blob:none --quiet https://github.com/Disty0/sdnq /tmp/pip-req-build-47u0yvzs
  Resolved https://github.com/Disty0/sdnq to commit db8fad818997c259c973a04b6e6963aa8561c170
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
import diffusers
from sdnq import SDNQConfig # import sdnq to register it into diffusers and transformers
from sdnq.loader import apply_sdnq_options_to_model

pipe = diffusers.ZImagePipeline.from_pretrained("Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32", torch_dtype=torch.float32, device_map="cuda")
pipe.transformer = apply_sdnq_options_to_model(pipe.transformer, use_quantized_matmul=True)
pipe.text_encoder = apply_sdnq_options_to_model(pipe.text_encoder, use_quantized_matmul=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1406 [00:00<?, ?it/s]

In [3]:
# prompt = "a grizzled 60-year-old mage, his face a striking fusion of ancient mysticism and cutting-edge cyberware, staring directly into the camera with piercing, bioluminescent eyes that flicker between arcane violet and cold machine blue. His silver-streaked beard is woven with delicate gold circuitry, pulsing faintly with energy, while the left side of his face transitions seamlessly into sleek, blackened metal plating, etched with glowing runes that hum with latent power. His robe, a tattered mix of enchanted fabric and nano-weave armor, clings to his broad shoulders, its frayed edges crackling with unstable magic. Behind him, a sprawling microchip cityscape throbs with neon-yellow circuit patterns against an abyssal black void, the labyrinthine pathways mirroring the intricate scars and implants across his weathered skin. The air around him shimmers with distortion—part holographic spell matrix, part overheating processor—as if reality itself struggles to contain him."
# image = pipe(
#     prompt=prompt,
#     height=1024,
#     width=1024,
#     num_inference_steps=9,
#     guidance_scale=0.0,
#     generator=torch.manual_seed(42),
# ).images[0]
# display(image)

In [4]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [5]:
!pip install uvicorn


In [6]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from io import BytesIO
import base64
import torch
import uvicorn
import threading
import time

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"])

TOKEN = 'cs-secret-2026'

class Req(BaseModel):
    prompt: str

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/generate")
async def generate(req: Req, x_token: str = None):
    if x_token != TOKEN:
        raise HTTPException(status_code=401, detail="Invalid")
    image = pipe(prompt=req.prompt, height=1024, width=1024,
                 num_inference_steps=9, guidance_scale=0.0,
                 generator=torch.manual_seed(42)).images[0]
    buf = BytesIO()
    image.save(buf, format="PNG")
    return {"image": base64.b64encode(buf.getvalue()).decode()}

# Mata qualquer processo que esteja usando a porta 8081
!fuser -k 8081/tcp

def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8081, log_level="info")
    server = uvicorn.Server(config)
    server.run()

# Inicia em Thread para compartilhar memória (necessário para acessar 'pipe' e 'app')
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(2)
print("🚀 Servidor iniciado via Thread na porta 8081")

INFO:     Started server process [28635]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8081 (Press CTRL+C to quit)


🚀 Servidor iniciado via Thread na porta 8081


In [7]:
import subprocess, re, time

CLOUDFLARE_TOKEN = "eyJhIjoiNmIxNmYwMzUwNjM5NWFhNjBjZjk2NzY0MDA2Y2I0MGUiLCJ0IjoiYzA1MzQ2NGEtNzBhYy00MmUwLWFkMjQtZDdiMDFkYzBmOGZhIiwicyI6Ik1qbGtOekEyWVRjdFkyWXdOQzAwTXpGa0xXSTNZV1V0WkdJek5HVTJNRGhrT0RVeCJ9"

cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel',  '--url', 'http://localhost:8081'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

tunnel_url = None
print('⏳ Esperando URL do tunnel...')
for i in range(30):
    line = cloudflared_proc.stdout.readline().decode('utf-8', errors='replace')
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f"\n🌐 TÚNEL: {tunnel_url}")
else:
    print("❌ Não encontrou URL")


⏳ Esperando URL do tunnel...
2026-05-10T02:10:05Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-10T02:10:05Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-10T02:10:10Z INF +--------------------------------------------------------------------------------------------+
2026-05-10T02:10:10Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-10T02:10:10Z INF |  https://donated-imaginat

In [8]:
import os

print("--- Conteúdo do uvicorn.log ---")
if os.path.exists("uvicorn.log"):
    with open("uvicorn.log", "r") as f:
        print(f.read())
else:
    print("Arquivo uvicorn.log não encontrado.")

print("\n--- Verificando processos ativos ---")
!ps aux | grep uvicorn | grep -v grep

print("\n--- Verificando portas ---")
!ss -tlnp | grep 8081

--- Conteúdo do uvicorn.log ---
ERROR:    Error loading ASGI app. Attribute "app" not found in module "__main__".


--- Verificando processos ativos ---

--- Verificando portas ---
LISTEN 0      2048         0.0.0.0:8081       0.0.0.0:*    users:(("python3",pid=28635,fd=104))   
